# **Problema 2: Integración numérica de flujos**

Considere los siguientes campos de velocidad dependientes del tiempo $\mathbf{u}(\mathbf{x}, t)$ en el caso bidimensional $\mathbf{x} = x~\mathbf{\hat{x}}+ y~\mathbf{\hat{y}}$.
En cada caso, adimensionalice los campos y calcule numéricamente las trayectorias y la evolución de las líneas de corriente eligiendo puntos de colocación iniciales más apropiados para el flujo en cuestión:

(i) Una fuente lineal de caudal constante $Q$ superpuesta a una corriente uniforme cuya velocidad aumenta linealmente con el tiempo $U(t) = \gamma_0 t$.

(ii) Una fuente lineal de caudal constante $Q$ superpuesta a un torbellino cuya circulación $\Gamma$ decrece exponencialmente con un tiempo característico $\tau$.
_Ayuda: Tras adimensionalizar quedará un parámetro libre. Bárralo para estudiar los distintos regímenes._

(iii) Una fuente lineal de caudal constante $Q$ superpuesta a una corriente uniforme oscilante con velocidad $U(t) = U_0\cos(\omega t)$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 2. Motor Numérico y Modelo Físico ---
def esquema(X, u, t, dt):
    return X + u(X, t) * dt

def linea_de_corriente(l0, u, t, N, ds):
    d = l0.shape[0]
    s = np.arange(0, N+1)*ds
    ls = np.zeros((N+1, d))
    ls[0] = l0

    u_lc = lambda x, _: u(x, t)
    for j in range(N):
        ls[j+1] = esquema(ls[j], u_lc, t, ds)
    return s, ls

def campo_adimensional(X, t):
    x, y = X[0], X[1]
    r2 = x**2 + y**2
    if r2 < 1e-4: # Manejo de singularidad
        return np.array([0.0, 0.0])
    u_x = (x / r2) + t
    u_y = (y / r2)
    return np.array([u_x, u_y])

# --- 3. Condiciones Iniciales Estratégicas ---
thetas = np.linspace(0, 2*np.pi, 20, endpoint=False)
radio_emision = 0.05
l0s_fuente = [np.array([radio_emision*np.cos(th), radio_emision*np.sin(th)]) for th in thetas]
l0s_borde = [np.array([-1.5, y]) for y in np.linspace(-1.5, 1.5, 15)]
l0s = np.array(l0s_fuente + l0s_borde)

N_pasos = 400
ds = 0.01

# --- 4. Función de renderizado ---
def graficar_frame(t_val):
    plt.figure(figsize=(7, 7))

    for l0 in l0s:
        _, ls = linea_de_corriente(l0, campo_adimensional, t_val, N_pasos, ds)
        plt.plot(ls[:,0], ls[:,1], color='blue', alpha=0.6)

    # Marcador de la singularidad como PUNTO ('ro')
    plt.plot(0, 0, 'ro', markersize=6, label='Fuente (Singularidad)')

    plt.xlim(-1.5, 1.5)
    plt.ylim(-1.5, 1.5)
    plt.xlabel(r'Coordenada adimensional $\tilde{x}$')
    plt.ylabel(r'Coordenada adimensional $\tilde{y}$')
    plt.title(f'Evolución Topológica de Líneas de Corriente a $\\tilde{{t}} = {t_val:.2f}$')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.show()


# Para visualizar en diferentes tiempos, puedes llamar a graficar_frame con diferentes valores
graficar_frame(0.0)
graficar_frame(1.0)
graficar_frame(2.0)


# Ahora calculamos las trayectorias..

import numpy as np
import matplotlib.pyplot as plt

# --- Motor Numérico Original ---
def esquema(X, u, t, dt):
    return X + u(X, t) * dt

def trayectoria(X0, u, N, dt):
    d = X0.shape[0]
    ts = np.arange(0, N+1)*dt
    Xs = np.zeros((N+1, d))
    Xs[0] = X0
    for j in range(N):
        Xs[j+1] = esquema(Xs[j], u, ts[j], dt)
    return ts, Xs

# --- Campo Físico Estabilizado ---
def campo_adimensional_seguro(X, t):
    x, y = X[0], X[1]
    r2 = x**2 + y**2

    # Parámetro de suavización (Softening) para evitar la explosión de Euler
    epsilon = 0.05
    denom = r2 + epsilon**2

    u_x = (x / denom) + t
    u_y = (y / denom)
    return np.array([u_x, u_y])

# --- Tu Esqueleto Corregido ---
# Aumentamos N a 400 para darle tiempo a la corriente de arrastrar
# a las partículas de vuelta hacia el dominio visible.
N = 400
dt = 1e-2

# Barrera de inyección en L
l0s = [[-1, a] for a in np.linspace(-1, 1, 9)]
l0s = l0s + [[a, -1] for a in np.linspace(-1, 1, 9)]
l0s = np.array(l0s)

plt.figure(figsize=(8, 8))

for i in range(l0s.shape[0]):
    # Llamamos a tu integrador
    t_arr, Xs = trayectoria(l0s[i], campo_adimensional_seguro, N, dt)

    # Graficamos según tu convención
    plt.plot(Xs[0,0], Xs[0,1], "ko", zorder=3)
    plt.plot(Xs[:,0], Xs[:,1], "b-", alpha=0.7)

# Marcamos el origen (la fuente suavizada)
plt.plot(0, 0, 'rX', markersize=10, label='Fuente (Singularidad)')

# Ampliamos la ventana para poder ver el "turnaround" (el retroceso)
plt.xlim(-2.5, 2)
plt.ylim(-2, 2)
plt.xlabel(r'$\tilde{x}$')
plt.ylabel(r'$\tilde{y}$')
plt.title('Trayectorias (Euler Estabilizado)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()


